In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

In [ ]:
news = pd.read_csv("../data/raw/news.csv")
stocks = pd.read_csv("../data/processed/stocks_with_indicators.csv")  # or your saved file

In [ ]:
news["date"] = pd.to_datetime(news["date"], utc=True)
stocks["Date"] = pd.to_datetime(stocks["Date"])

In [ ]:
nltk.download("vader_lexicon")

In [ ]:
sia = SentimentIntensityAnalyzer()

In [ ]:
def get_sentiment(text):
    return sia.polarity_scores(str(text))["compound"]

news["sentiment"] = news["headline"].apply(get_sentiment)

In [ ]:
news["date"] = news["date"].dt.date
stocks["Date_only"] = stocks["Date"].dt.date

In [ ]:
daily_sentiment = news.groupby(["stock", "date"])["sentiment"].mean().reset_index()
daily_sentiment.rename(columns={"date": "Date_only"}, inplace=True)

In [ ]:
stocks["Ticker"] = stocks["Ticker"].str.upper()
daily_sentiment["stock"] = daily_sentiment["stock"].str.upper()

In [ ]:
merged = pd.merge(
    stocks,
    daily_sentiment,
    left_on=["Ticker", "Date_only"],
    right_on=["stock", "Date_only"],
    how="inner"
)

In [ ]:
merged = merged.dropna(subset=["sentiment", "Return"])

In [ ]:
correlation = merged["sentiment"].corr(merged["Return"])
correlation

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(merged["sentiment"], merged["Return"], alpha=0.5)

plt.title(f"Sentiment vs Stock Returns (Corr={correlation:.3f})")
plt.xlabel("News Sentiment")
plt.ylabel("Daily Return (%)")
plt.show()

In [ ]:
def label_sentiment(x):
    if x > 0.05:
        return "Positive"
    elif x < -0.05:
        return "Negative"
    else:
        return "Neutral"

merged["sentiment_class"] = merged["sentiment"].apply(label_sentiment)

In [ ]:
grouped = merged.groupby("sentiment_class")["Return"].mean()
grouped

In [ ]:
grouped.plot(kind="bar", figsize=(7,4))
plt.title("Average Returns by Sentiment Class")
plt.show()